# 05. Client Profitability

## Business Question

Which clients generate the highest profit rather than just revenue?

### Methodology Note

To ensure fair comparison, the analysis focuses on clients active during 2024. Comparing clients across incomplete business periods (e.g., partial 2022 or 2024 activity) could lead to misleading conclusions.

Overall business performance (all periods, all clients) is covered in `04_business_overview.ipynb`; this notebook does not repeat it.

In [1]:
import os
from pathlib import Path

import pandas as pd

from dotenv import load_dotenv
from sqlalchemy import create_engine, text

In [2]:
load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
    pool_pre_ping=True,
    future=True
)

try:
    with engine.connect() as connection:
        connection.execute(text("SELECT 1"))
    print("Successfully connected to MySQL.")
except Exception as e:
    print(f"Connection failed:\n{e}")

Successfully connected to MySQL.


### Load Data

`trip_analytical_dataset` (all trips, all statuses) joined with `is_seasonal` from `clients`.

In [3]:
trips = pd.read_sql(
    text("SELECT * FROM trip_analytical_dataset"),
    engine,
    parse_dates=["date_departure", "date_arrival"]
)

clients = pd.read_sql(
    text("SELECT client_id, is_seasonal FROM clients"),
    engine
)

trips = trips.merge(clients, on="client_id", how="left")

print(f"Rows loaded: {len(trips):,}")

Rows loaded: 4,925


## Dataset Validation

In [4]:
print(f"Total trips: {len(trips):,}")
print(f"Unique clients: {trips['client_id'].nunique()}")
print(f"Duplicate trip_id: {trips['trip_id'].duplicated().sum()}")
print(f"Trips with revenue and fuel cost present: {trips.dropna(subset=['revenue_uah', 'fuel_cost_uah']).shape[0]:,}")

Total trips: 4,925
Unique clients: 10
Duplicate trip_id: 0
Trips with revenue and fuel cost present: 4,827


## 2024 Active Clients Filter

Only clients with at least one trip in 2024 are included below.

In [5]:
trips_2024 = trips[trips["date_departure"].dt.year == 2024]
active_clients_2024 = trips_2024["client_id"].unique()

all_clients = trips["client_id"].nunique()
active_clients_count = len(active_clients_2024)

print(f"Total clients (all periods): {all_clients}")
print(f"Active in 2024: {active_clients_count}")
print(f"Excluded (inactive in 2024): {all_clients - active_clients_count}")

valid_2024 = trips_2024[
    trips_2024["client_id"].isin(active_clients_2024)
].dropna(subset=["revenue_uah", "fuel_cost_uah"])

print(f"Valid 2024 trips (revenue + fuel cost present): {len(valid_2024):,}")

Total clients (all periods): 10
Active in 2024: 10
Excluded (inactive in 2024): 0
Valid 2024 trips (revenue + fuel cost present): 1,362


## Client Profitability Table

`low_sample` flags clients with fewer than 15 trips in 2024 — their margin should be interpreted cautiously and they are excluded from the Low-Margin Clients deep dive below.

In [6]:
def summarize(df, group_cols):
    g = df.groupby(group_cols).agg(
        trip_count=("trip_id", "count"),
        revenue_uah=("revenue_uah", "sum"),
        fuel_cost_uah=("fuel_cost_uah", "sum"),
    )
    g["gross_profit_uah"] = g["revenue_uah"] - g["fuel_cost_uah"]
    g["gross_margin_pct"] = (g["gross_profit_uah"] / g["revenue_uah"] * 100).round(1)
    return g.reset_index()

In [7]:
client_profitability_2024 = summarize(valid_2024, ["client_id", "client_name", "is_seasonal"])
client_profitability_2024["low_sample"] = client_profitability_2024["trip_count"] < 15
client_profitability_2024 = client_profitability_2024.sort_values("gross_profit_uah", ascending=False)

client_profitability_2024

,client_id,client_name,is_seasonal,trip_count,revenue_uah,fuel_cost_uah,gross_profit_uah,gross_margin_pct,low_sample
2,CLT-03,ТОВ Полтавські Лани,0,167,4.141124e+06,2056267.96,2.084856e+06,50.3,False
1,CLT-02,ПП ДніпроЗерно,0,192,3.889903e+06,1855357.06,2.034546e+06,52.3,False
0,CLT-01,ТОВ Соняшник-Агро,0,227,3.064563e+06,1176823.46,1.887739e+06,61.6,False
5,CLT-06,АгроЛогістика Центр,0,184,2.537897e+06,1036088.39,1.501809e+06,59.2,False
4,CLT-05,ТОВ Соєвий Дім,0,94,7.281712e+05,236643.35,4.915278e+05,67.5,False
7,CLT-08,ТОВ Пшеничний Шлях,1,108,7.590143e+05,346582.00,4.124323e+05,54.3,False
3,CLT-04,ФГ Кукурудза Південь,0,107,6.616704e+05,314586.28,3.470841e+05,52.5,False
9,CLT-11,ТОВ Нива Експорт,1,90,7.608691e+05,427764.19,3.331049e+05,43.8,False
8,CLT-09,ФГ Золоте Колосся,1,100,5.162626e+05,244008.27,2.722543e+05,52.7,False
6,CLT-07,ФГ Жнива-Поділля,1,93,2.230247e+05,104913.66,1.181111e+05,53.0,False


#### Key Findings
All 10 clients exceed the 15-trip threshold (`low_sample = False` for all) — no client requires exclusion due to insufficient sample size.

## Top Clients by Profit

In [8]:
client_profitability_2024.head(3)[
    ["client_name", "trip_count", "revenue_uah", "gross_profit_uah", "gross_margin_pct"]
]

,client_name,trip_count,revenue_uah,gross_profit_uah,gross_margin_pct
2,ТОВ Полтавські Лани,167,4.141124e+06,2.084856e+06,50.3
1,ПП ДніпроЗерно,192,3.889903e+06,2.034546e+06,52.3
0,ТОВ Соняшник-Агро,227,3.064563e+06,1.887739e+06,61.6


#### Key Findings
- Top 3 clients (ТОВ Полтавські Лани, ПП ДніпроЗерно, ТОВ Соняшник-Агро) generate the majority of total 2024 gross profit — see Profit Concentration below.
- The highest-revenue client (ТОВ Полтавські Лани) has the lowest margin among the top 5 (50.3%) — its scale, not efficiency, drives its #1 profit rank.
- ТОВ Соняшник-Агро combines the highest trip volume (227) with a strong margin (61.6%) — the best volume-efficiency combination in the portfolio.

## Profit Concentration

In [9]:
total_profit_2024 = client_profitability_2024["gross_profit_uah"].sum()

concentration = client_profitability_2024.copy()
concentration["profit_share_pct"] = (concentration["gross_profit_uah"] / total_profit_2024 * 100).round(1)
concentration["cumulative_share_pct"] = concentration["profit_share_pct"].cumsum().round(1)

top3_share = concentration.head(3)["profit_share_pct"].sum().round(1)
print(f"Top 3 clients generate {top3_share}% of total 2024 gross profit.")

concentration[["client_name", "gross_profit_uah", "profit_share_pct", "cumulative_share_pct"]].head(6)

Top 3 clients generate 63.4% of total 2024 gross profit.


,client_name,gross_profit_uah,profit_share_pct,cumulative_share_pct
2,ТОВ Полтавські Лани,2.084856e+06,22.0,22.0
1,ПП ДніпроЗерно,2.034546e+06,21.5,43.5
0,ТОВ Соняшник-Агро,1.887739e+06,19.9,63.4
5,АгроЛогістика Центр,1.501809e+06,15.8,79.2
4,ТОВ Соєвий Дім,4.915278e+05,5.2,84.4
7,ТОВ Пшеничний Шлях,4.124323e+05,4.3,88.7


#### Key Findings

- Top 3 clients (ТОВ Полтавські Лани, ПП ДніпроЗерно, ТОВ Соняшник-Агро) generate 63.4% of total 2024 gross profit.
- Half of the client base (5 of 10 clients) accounts for 84.4% of total profit — moderate concentration, softer than a classic 80/20 Pareto split.
- The remaining 5 clients together contribute only 15.6% of total profit, despite each requiring full operational and administrative overhead.
- Evaluate whether the bottom 5 clients (15.6% combined profit share) justify their share of fleet capacity and administrative overhead, or whether reallocating capacity toward the top 5 clients would improve overall profitability.

## Low-Margin Clients

Excludes `low_sample` clients (fewer than 15 trips in 2024).

In [10]:
low_margin_clients = client_profitability_2024[
    ~client_profitability_2024["low_sample"]
].sort_values("gross_margin_pct").head(3)

low_margin_clients[
    ["client_name", "trip_count", "revenue_uah", "gross_profit_uah", "gross_margin_pct"]
]

,client_name,trip_count,revenue_uah,gross_profit_uah,gross_margin_pct
9,ТОВ Нива Експорт,90,7.608691e+05,3.331049e+05,43.8
2,ТОВ Полтавські Лани,167,4.141124e+06,2.084856e+06,50.3
1,ПП ДніпроЗерно,192,3.889903e+06,2.034546e+06,52.3


## Deep Dive

Operational and route profile for the low-margin clients identified above — checking whether seasonality, route length, load factor, or reliability explain the lower margin.

In [11]:
low_margin_ids = low_margin_clients["client_id"]

deep_dive_valid = valid_2024[valid_2024["client_id"].isin(low_margin_ids)]
deep_dive_full = trips_2024[trips_2024["client_id"].isin(low_margin_ids)]

route_mix = deep_dive_valid.groupby(["client_id", "route_type"]).size().unstack(fill_value=0)
route_mix_share = route_mix.div(route_mix.sum(axis=1), axis=0).round(2)

operational_profile = deep_dive_valid.groupby("client_id").agg(
    avg_distance_km=("trip_distance_km", "mean"),
    avg_load_factor_pct=("load_factor_pct", "mean"),
).round(1)

status_counts = deep_dive_full.groupby(["client_id", "status"]).size().unstack(fill_value=0)
status_counts["total"] = status_counts.sum(axis=1)
status_counts["cancelled_pct"] = (status_counts.get("cancelled", 0) / status_counts["total"] * 100).round(1)
status_counts["delayed_pct"] = (status_counts.get("delayed", 0) / status_counts["total"] * 100).round(1)

deep_dive = (
    low_margin_clients[["client_id", "client_name", "is_seasonal", "gross_margin_pct"]]
    .merge(operational_profile, on="client_id")
    .merge(status_counts[["cancelled_pct", "delayed_pct"]], on="client_id")
    .merge(route_mix_share, on="client_id")
)

deep_dive

,client_id,client_name,is_seasonal,gross_margin_pct,avg_distance_km,avg_load_factor_pct,cancelled_pct,delayed_pct,highway
0,CLT-11,ТОВ Нива Експорт,1,43.8,214.0,85.6,0.0,21.1,1.0
1,CLT-03,ТОВ Полтавські Лани,0,50.3,612.0,90.7,0.0,29.5,1.0
2,CLT-02,ПП ДніпроЗерно,0,52.3,472.0,90.6,0.5,27.7,1.0


#### Key Findings

- All three lowest-margin clients operate 100% highway (long-distance) routes with elevated delay rates (21-30%, above the company average) — the primary driver of their lower margin is idle fuel consumption during delays on long routes, not seasonality or load factor.

## Follow-Up: Structural Risk Clients Deep Dive

Two of the three low-margin clients — ТОВ Полтавські Лани and ПП ДніпроЗерно — combine a large share of total profit with below-median margin and elevated delay rates. For both, the underlying cause is separated into two hypotheses: tariff-driven or delay-driven.

ТОВ Нива Експорт has the lowest margin overall but a small profit share (7.5%) — here the question is different: is this a new client still ramping up, or an established, persistently low-performing relationship?

In [12]:
followup_ids = ["CLT-03", "CLT-02"]  # ТОВ Полтавські Лани, ПП ДніпроЗерно

top_clients_ids = client_profitability_2024.head(5)["client_id"]

rate_comparison = (
    valid_2024[valid_2024["client_id"].isin(top_clients_ids)]
    .assign(effective_rate=lambda d: d["revenue_uah"] / (d["trip_distance_km"] * d["cargo_tons_actual"]))
    .groupby("client_id")["effective_rate"]
    .mean()
    .round(2)
    .sort_values()
)

rate_comparison = rate_comparison.reset_index().merge(
    client_profitability_2024[["client_id", "client_name"]],
    on="client_id"
)

for client_id in followup_ids:
    name = rate_comparison.loc[rate_comparison["client_id"] == client_id, "client_name"].values[0]
    rate = rate_comparison.loc[rate_comparison["client_id"] == client_id, "effective_rate"].values[0]
    print(f"{name}: {rate} UAH/ton-km")

print(f"\nRange among top 5 clients: {rate_comparison['effective_rate'].min()} - {rate_comparison['effective_rate'].max()} UAH/ton-km")

rate_comparison[["client_name", "effective_rate"]]

ТОВ Полтавські Лани: 1.87 UAH/ton-km
ПП ДніпроЗерно: 1.98 UAH/ton-km

Range among top 5 clients: 1.87 - 2.95 UAH/ton-km


,client_name,effective_rate
0,ТОВ Полтавські Лани,1.87
1,ПП ДніпроЗерно,1.98
2,АгроЛогістика Центр,2.40
3,ТОВ Соняшник-Агро,2.54
4,ТОВ Соєвий Дім,2.95


Both structural-risk clients have below-average tariffs among the top-5 clients: ТОВ Полтавські Лани at 1.87 UAH/ton-km (the lowest of all five) and ПП ДніпроЗерно at 1.98 — both sitting well below АгроЛогістика Центр (2.40), ТОВ Соняшник-Агро (2.54), and ТОВ Соєвий Дім (2.95). This is a 21-37% gap versus the highest-paying comparable clients, confirming the margin issue is partly tariff-driven, not solely a byproduct of delays or route length.

In [13]:
followup_ids = ["CLT-02", "CLT-03"]  # ПП ДніпроЗерно, ТОВ Полтавські Лани

followup_delays = trips_2024[
    (trips_2024["client_id"].isin(followup_ids)) & (trips_2024["status"] == "delayed")
]

followup_delays.groupby(["client_id", "origin_city", "destination_city"]).agg(
    delay_count=("trip_id", "count"),
    avg_delay_hours=("delay_hours", "mean"),
).sort_values(["client_id", "delay_count"], ascending=[True, False])

,,,delay_count,avg_delay_hours
client_id,origin_city,destination_city,,
CLT-02,Дніпро,Одеса,56,28.408929
CLT-03,Полтава,Чорноморськ,51,26.125490


Both structural-risk clients are underpaying relative to peers — ТОВ Полтавські Лани (1.87 UAH/ton-km) and ПП ДніпроЗерно (1.98) sit at the bottom of the top-5 rate range (1.87-2.95). Their delays are also fully port-concentrated: 51/51 delayed trips on Полтава → Чорноморськ (avg. 26.1h) and 56/56 on Дніпро → Одеса (avg. 28.4h) — two independent clients, same pattern, both pointing to port-side bottlenecks rather than random operational noise.

### ТОВ Нива Експорт — New Client or Persistent Underperformer?

In [14]:
niva_id = "CLT-11"

niva_tenure = trips[trips["client_id"] == niva_id]["date_departure"].agg(["min", "max"])
print(f"First trip: {niva_tenure['min'].date()}")
print(f"Last trip: {niva_tenure['max'].date()}")

First trip: 2022-07-03
Last trip: 2024-09-30


In [15]:
niva_valid = valid_2024[valid_2024["client_id"] == niva_id].copy()
niva_valid["quarter"] = niva_valid["date_departure"].dt.to_period("Q")

niva_quarterly = niva_valid.groupby("quarter").agg(
    trip_count=("trip_id", "count"),
    revenue_uah=("revenue_uah", "sum"),
    fuel_cost_uah=("fuel_cost_uah", "sum"),
)
niva_quarterly["gross_margin_pct"] = (
    (niva_quarterly["revenue_uah"] - niva_quarterly["fuel_cost_uah"]) / niva_quarterly["revenue_uah"] * 100
).round(1)

niva_quarterly

,trip_count,revenue_uah,fuel_cost_uah,gross_margin_pct
quarter,,,,
2024Q3,90,760869.1034,427764.19,43.8


In [16]:
niva_seasonal_trend = (
    trips[
        (trips["client_id"] == niva_id)
        & (trips["is_seasonal"] == 1)
    ]
    .dropna(subset=["revenue_uah", "fuel_cost_uah"])
    .assign(year=lambda d: d["date_departure"].dt.year)
    .groupby("year")
    .agg(
        trip_count=("trip_id", "count"),
        revenue_uah=("revenue_uah", "sum"),
        fuel_cost_uah=("fuel_cost_uah", "sum"),
    )
)

niva_seasonal_trend["gross_margin_pct"] = (
    (niva_seasonal_trend["revenue_uah"] - niva_seasonal_trend["fuel_cost_uah"]) / niva_seasonal_trend["revenue_uah"] * 100
).round(1)

niva_seasonal_trend

,trip_count,revenue_uah,fuel_cost_uah,gross_margin_pct
year,,,,
2022,83,654168.3610,333750.11,49.0
2023,102,883842.2346,431414.98,51.2
2024,90,760869.1034,427764.19,43.8


- **ТОВ Нива Експорт** has the lowest margin in 2024 (43.8%) and the highest delay rate in the portfolio, but contributes only 7.5% of total profit. The client is not new — active since 2022-07-03, with a margin history of 49.0% (2022) → 51.2% (2023, peak) → 43.8% (2024). This represents a genuine decline in 2024, not a chronic issue or an early ramp-up period, and is consistent with the elevated delay rate observed this year. Recommended action: investigate whether 2024-specific delays (route, timing, or operational changes) explain the drop before deciding on tariff or relationship changes — the 2022-2023 performance suggests the relationship was previously healthy and may be recoverable.

## Business Recommendations

**ТОВ Полтавські Лани — two concrete, independent actions:**
  1. Renegotiate the tariff toward the 2.0-2.5 UAH/ton-km range paid by comparable top clients — the current 1.87 rate is below-market for this client's volume and distance profile.
  2. Investigate port-side delays specifically on the Полтава → Чорноморськ route (51 of 51 delayed trips, avg. 26.1 hours) — this is a single, addressable bottleneck, not a company-wide issue.

**ПП ДніпроЗерно — same structural pattern, same two-step review:**
  1. Tariff sits at 1.98 UAH/ton-km — closer to market than Полтавські Лани, but still below the top-5 average (range: 1.87-2.95). Room for a smaller upward adjustment.
  2. Delays are equally concentrated on a single route — Дніпро → Одеса (56 of 56 delayed trips, avg. 28.4 hours) — another port-side bottleneck, and the second data point suggesting port routes in general merit a dedicated review (see `07_route_analysis.ipynb`).

**ТОВ Нива Експорт — investigate 2024 specifically, not the relationship as a whole:**
  Active since 2022-07-03, with margin history of 49.0% (2022) → 51.2% (2023, peak) → 43.8% (2024) — a genuine decline this year, not a chronic issue or an early ramp-up period, and consistent with this client's elevated 2024 delay rate. Investigate what changed operationally in 2024 before considering any tariff or relationship changes; the 2022-2023 track record suggests the relationship is recoverable.

**Portfolio-wide:**
- As a parallel strategy, increase business volume with high-margin, lower-volume clients (e.g. ТОВ Соєвий Дім, 67.5% margin) to reduce dependency on the less efficient top-revenue relationships and diversify profit sources.
- Monitor cancellation/delay rates for all flagged clients even where margin is currently acceptable, as reliability risk can erode profitability over time (see `03_build_trip_analytical_dataset.ipynb` delay-margin finding).

In [17]:
output_path = Path("../data/processed/client_profitability_2024.csv")
client_profitability_2024.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

Saved: ../data/processed/client_profitability_2024.csv


In [18]:
client_profitability_2024.to_sql("client_profitability_2024", engine, if_exists="replace", index=False)
print("client_profitability_2024 written to MySQL.")

client_profitability_2024 written to MySQL.
